<h3>Criar base da dados</h3>

In [23]:
import sqlite3

# Conecta (ou cria) o banco de dados
conn = sqlite3.connect('biblioteca_equipamentos.db')
cursor = conn.cursor()

# Habilita o suporte a chaves estrangeiras (o SQLite não faz isso por padrão)
cursor.execute("PRAGMA foreign_keys = ON;")

# Tabela ALUNO
cursor.execute("""
CREATE TABLE IF NOT EXISTS ALUNO (
    ID INTEGER PRIMARY KEY,
    NOME VARCHAR(100) NOT NULL
);
""")

try:
    cursor.execute("ALTER TABLE ALUNO ADD COLUMN BLOQUEADO_ATE DATE DEFAULT NULL;")
except sqlite3.OperationalError as e:
    print(f"Aviso: {e}")

# Tabela EQUIPAMENTO
cursor.execute("""
CREATE TABLE IF NOT EXISTS EQUIPAMENTO (
    ID INTEGER PRIMARY KEY,
    NOME VARCHAR(100) NOT NULL
);
""")

# Tabela EMPRESTIMO
cursor.execute("""
CREATE TABLE IF NOT EXISTS EMPRESTIMO (
    ID INTEGER PRIMARY KEY,
    ALUNO_ID INTEGER NOT NULL,
    EQUIPAMENTO_ID INTEGER NOT NULL,
    DATA_EMPRESTIMO DATE NOT NULL,
    DATA_PREVISTA DATE NOT NULL,
    DATA_DEVOLUCAO DATE DEFAULT NULL,
    FOREIGN KEY (ALUNO_ID) REFERENCES ALUNO(ID),
    FOREIGN KEY (EQUIPAMENTO_ID) REFERENCES EQUIPAMENTO(ID)
);
""")

# Confirma as alterações
conn.commit()

print("Tabelas criadas com sucesso!")

conn.close()

Tabelas criadas com sucesso!


<h3>Conectar ao banco de dados</h3>

In [24]:
DB_NAME = 'biblioteca_equipamentos.db'


def conectar():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

<h3>Funções para a entidade Aluno </h3>

In [25]:
def inserir_aluno(nome):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("INSERT INTO ALUNO (NOME) VALUES (?);", (nome,))
    conn.commit()
    novo_id = cursor.lastrowid
    conn.close()
    print(f"Aluno '{nome}' cadastrado com ID {novo_id}.")
    return novo_id


def listar_alunos():
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT ID, NOME FROM ALUNO ORDER BY ID;")
    alunos = cursor.fetchall()
    conn.close()

    print("\n--- ALUNOS CADASTRADOS ---")
    if not alunos:
        print("Nenhum aluno cadastrado.")
    for id_, nome in alunos:
        print(f"[{id_}] {nome}")


def buscar_aluno_por_id(id_aluno):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT ID, NOME FROM ALUNO WHERE ID = ?;", (id_aluno,))
    aluno = cursor.fetchone()
    conn.close()

    print("\n--- DADOS DO ALUNO ---")
    if aluno:
        print(f"ID: {aluno[0]}\nNome: {aluno[1]}")
    else:
        print(f"Nenhum aluno encontrado com ID {id_aluno}.")
    return aluno


def atualizar_aluno(id_aluno, novo_nome):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("UPDATE ALUNO SET NOME = ? WHERE ID = ?;", (novo_nome, id_aluno))
    conn.commit()
    linhas_afetadas = cursor.rowcount
    conn.close()

    if linhas_afetadas:
        print(f"Aluno {id_aluno} atualizado para '{novo_nome}'.")
    else:
        print(f"Nenhum aluno encontrado com ID {id_aluno}. Nada foi atualizado.")


def deletar_aluno(id_aluno):
    conn = conectar()
    cursor = conn.cursor()
    try:
        cursor.execute("DELETE FROM ALUNO WHERE ID = ?;", (id_aluno,))
        conn.commit()
        if cursor.rowcount:
            print(f"Aluno {id_aluno} removido com sucesso.")
        else:
            print(f"Nenhum aluno encontrado com ID {id_aluno}.")
    except sqlite3.IntegrityError:
        print(f"Não foi possível remover o aluno {id_aluno}: existem empréstimos vinculados a ele.")
    finally:
        conn.close()

<h3>Funções para entidade Equipamento </h3>

In [26]:
def inserir_equipamento(nome):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("INSERT INTO EQUIPAMENTO (NOME) VALUES (?);", (nome,))
    conn.commit()
    novo_id = cursor.lastrowid
    conn.close()
    print(f"Equipamento '{nome}' cadastrado com ID {novo_id}.")
    return novo_id

def listar_equipamentos():
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT ID, NOME FROM EQUIPAMENTO ORDER BY ID;")
    equipamentos = cursor.fetchall()
    conn.close()

    print("\n--- EQUIPAMENTOS CADASTRADOS ---")
    if not equipamentos:
        print("Nenhum equipamento cadastrado.")
    for id_, nome in equipamentos:
        print(f"[{id_}] {nome}")

def buscar_equipamento_por_id(id_equipamento):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT ID, NOME FROM EQUIPAMENTO WHERE ID = ?;", (id_equipamento,))
    equipamento = cursor.fetchone()
    conn.close()

    print("\n--- DADOS DO EQUIPAMENTO ---")
    if equipamento:
        print(f"ID: {equipamento[0]}\nNome: {equipamento[1]}")
    else:
        print(f"Nenhum equipamento encontrado com ID {id_equipamento}.")
    return equipamento


def atualizar_equipamento(id_equipamento, novo_nome):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("UPDATE EQUIPAMENTO SET NOME = ? WHERE ID = ?;", (novo_nome, id_equipamento))
    conn.commit()
    linhas_afetadas = cursor.rowcount
    conn.close()

    if linhas_afetadas:
        print(f"Equipamento {id_equipamento} atualizado para '{novo_nome}'.")
    else:
        print(f"Nenhum equipamento encontrado com ID {id_equipamento}. Nada foi atualizado.")


def deletar_equipamento(id_equipamento):
    conn = conectar()
    cursor = conn.cursor()
    try:
        cursor.execute("DELETE FROM EQUIPAMENTO WHERE ID = ?;", (id_equipamento,))
        conn.commit()
        if cursor.rowcount:
            print(f"Equipamento {id_equipamento} removido com sucesso.")
        else:
            print(f"Nenhum equipamento encontrado com ID {id_equipamento}.")
    except sqlite3.IntegrityError:
        print(f"Não foi possível remover o equipamento {id_equipamento}: existem empréstimos vinculados a ele.")
    finally:
        conn.close()

<h3>Funções de verificação para o empréstimo</h3>

In [27]:
from datetime import date

def aluno_existe(id_aluno):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT 1 FROM ALUNO WHERE ID = ?;", (id_aluno,))
    existe = cursor.fetchone() is not None
    conn.close()
    return existe


def equipamento_existe(id_equipamento):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT 1 FROM EQUIPAMENTO WHERE ID = ?;", (id_equipamento,))
    existe = cursor.fetchone() is not None
    conn.close()
    return existe


def equipamento_esta_emprestado(id_equipamento):
    """Regra: 'não queremos que os equipamentos sumam' -> item já emprestado não pode ser emprestado de novo."""
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT 1 FROM EMPRESTIMO
        WHERE EQUIPAMENTO_ID = ? AND DATA_DEVOLUCAO IS NULL;
    """, (id_equipamento,))
    resultado = cursor.fetchone() is not None
    conn.close()
    return resultado


def aluno_tem_emprestimo_atrasado(id_aluno):
    """Pendência tipo 1: empréstimo em aberto cuja DATA_PREVISTA já passou (base: data atual)."""
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT ID, EQUIPAMENTO_ID, DATA_PREVISTA
        FROM EMPRESTIMO
        WHERE ALUNO_ID = ?
          AND DATA_DEVOLUCAO IS NULL
          AND DATA_PREVISTA < date('now');
    """, (id_aluno,))
    atrasos = cursor.fetchall()
    conn.close()
    return atrasos


def aluno_esta_bloqueado_por_atraso(id_aluno):
    """Pendência tipo 2: aluno devolveu algo atrasado e ainda está dentro dos 7 dias de bloqueio."""
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT BLOQUEADO_ATE FROM ALUNO WHERE ID = ?;", (id_aluno,))
    resultado = cursor.fetchone()
    conn.close()

    if resultado is None:
        return None  # aluno não existe

    bloqueado_ate = resultado[0]
    if bloqueado_ate and bloqueado_ate >= date.today().isoformat():
        return bloqueado_ate
    return None


def aluno_tem_pendencia(id_aluno):
    """
    Combina as duas regras de pendência do texto:
    1) empréstimo em aberto e atrasado
    2) bloqueio de 7 dias por devolução atrasada anterior
    Retorna (tem_pendencia: bool, motivo: str ou None)
    """
    atrasos = aluno_tem_emprestimo_atrasado(id_aluno)
    if atrasos:
        detalhes = ", ".join(f"empréstimo {a[0]} (equip. {a[1]}, previsto {a[2]})" for a in atrasos)
        return True, f"possui empréstimo(s) em aberto e atrasado(s): {detalhes}"

    bloqueado_ate = aluno_esta_bloqueado_por_atraso(id_aluno)
    if bloqueado_ate:
        return True, f"está bloqueado até {bloqueado_ate} por ter devolvido um item com atraso"

    return False, None

<h3>Funções de regra de negócio do empréstimo</h3>

In [28]:
def emprestar_equipamento(id_aluno, id_equipamento):
    if not aluno_existe(id_aluno):
        print(f"Erro: aluno com ID {id_aluno} não existe.")
        return False

    if not equipamento_existe(id_equipamento):
        print(f"Erro: equipamento com ID {id_equipamento} não existe.")
        return False

    if equipamento_esta_emprestado(id_equipamento):
        print(f"Erro: o equipamento {id_equipamento} já está emprestado a outro aluno.")
        return False

    tem_pendencia, motivo = aluno_tem_pendencia(id_aluno)
    if tem_pendencia:
        print(f"Erro: aluno {id_aluno} não pode pegar equipamento — {motivo}.")
        return False

    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO EMPRESTIMO (ALUNO_ID, EQUIPAMENTO_ID, DATA_EMPRESTIMO, DATA_PREVISTA, DATA_DEVOLUCAO)
        VALUES (?, ?, date('now'), date('now', '+7 days'), NULL);
    """, (id_aluno, id_equipamento))
    conn.commit()
    novo_id = cursor.lastrowid
    conn.close()
    print(f"Empréstimo {novo_id} registrado: aluno {id_aluno} pegou equipamento {id_equipamento}. Prazo: 7 dias.")
    return novo_id

<h3>Devolução dos empréstimos</h3>

In [29]:
def devolver_equipamento(id_emprestimo):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT ALUNO_ID, DATA_PREVISTA, DATA_DEVOLUCAO
        FROM EMPRESTIMO WHERE ID = ?;
    """, (id_emprestimo,))
    resultado = cursor.fetchone()

    if resultado is None:
        print(f"Erro: empréstimo {id_emprestimo} não encontrado.")
        conn.close()
        return False

    id_aluno, data_prevista, data_devolucao = resultado

    if data_devolucao is not None:
        print(f"Erro: o empréstimo {id_emprestimo} já foi devolvido em {data_devolucao}.")
        conn.close()
        return False

    # Registra a devolução
    cursor.execute("UPDATE EMPRESTIMO SET DATA_DEVOLUCAO = date('now') WHERE ID = ?;", (id_emprestimo,))

    # Verifica se a devolução foi em atraso (comparando com a data prevista)
    cursor.execute("SELECT date('now') > ?;", (data_prevista,))
    foi_atrasado = cursor.fetchone()[0] == 1

    if foi_atrasado:
        cursor.execute("""
            UPDATE ALUNO SET BLOQUEADO_ATE = date('now', '+7 days') WHERE ID = ?;
        """, (id_aluno,))
        print(f"Devolução do empréstimo {id_emprestimo} registrada COM ATRASO (previsto: {data_prevista}).")
        print(f"Aluno {id_aluno} está bloqueado para novos empréstimos pelos próximos 7 dias.")
    else:
        print(f"Devolução do empréstimo {id_emprestimo} registrada dentro do prazo.")

    conn.commit()
    conn.close()
    return True

<h3>Relatório com métricas</h3>

In [30]:
def listar_emprestimos():
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT E.ID, A.NOME, Q.NOME, E.DATA_EMPRESTIMO, E.DATA_PREVISTA, E.DATA_DEVOLUCAO
        FROM EMPRESTIMO E
        JOIN ALUNO A ON A.ID = E.ALUNO_ID
        JOIN EQUIPAMENTO Q ON Q.ID = E.EQUIPAMENTO_ID
        ORDER BY E.ID;
    """)
    linhas = cursor.fetchall()
    conn.close()

    print("\n--- HISTÓRICO DE EMPRÉSTIMOS ---")
    if not linhas:
        print("Nenhum empréstimo registrado.")
    for id_, aluno, equip, data_emp, data_prev, data_dev in linhas:
        status = "DEVOLVIDO" if data_dev else "EM ABERTO"
        print(f"[{id_}] {aluno} -> {equip} | pegou {data_emp} | previsto {data_prev} | devolvido {data_dev or '-'} | {status}")


def listar_emprestimos_ativos():
    """Regra: 'queremos saber o que está emprestado e para quem'."""
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT E.ID, A.NOME, Q.NOME, E.DATA_EMPRESTIMO, E.DATA_PREVISTA
        FROM EMPRESTIMO E
        JOIN ALUNO A ON A.ID = E.ALUNO_ID
        JOIN EQUIPAMENTO Q ON Q.ID = E.EQUIPAMENTO_ID
        WHERE E.DATA_DEVOLUCAO IS NULL
        ORDER BY E.DATA_PREVISTA;
    """)
    linhas = cursor.fetchall()
    conn.close()

    hoje = date.today().isoformat()
    print("\n--- EQUIPAMENTOS EMPRESTADOS NO MOMENTO ---")
    if not linhas:
        print("Nenhum equipamento emprestado no momento.")
    for id_, aluno, equip, data_emp, data_prev in linhas:
        atrasado = " (ATRASADO)" if data_prev < hoje else ""
        print(f"[Empréstimo {id_}] {equip} está com {aluno} | pegou {data_emp} | previsto {data_prev}{atrasado}")


def buscar_emprestimo_por_id(id_emprestimo):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT E.ID, A.NOME, Q.NOME, E.DATA_EMPRESTIMO, E.DATA_PREVISTA, E.DATA_DEVOLUCAO
        FROM EMPRESTIMO E
        JOIN ALUNO A ON A.ID = E.ALUNO_ID
        JOIN EQUIPAMENTO Q ON Q.ID = E.EQUIPAMENTO_ID
        WHERE E.ID = ?;
    """, (id_emprestimo,))
    linha = cursor.fetchone()
    conn.close()

    print("\n--- DETALHES DO EMPRÉSTIMO ---")
    if not linha:
        print(f"Nenhum empréstimo encontrado com ID {id_emprestimo}.")
        return None
    id_, aluno, equip, data_emp, data_prev, data_dev = linha
    status = "DEVOLVIDO" if data_dev else "EM ABERTO"
    print(f"ID: {id_}\nAluno: {aluno}\nEquipamento: {equip}\nData empréstimo: {data_emp}\nPrevisão: {data_prev}\nDevolução: {data_dev or '-'}\nStatus: {status}")
    return linha


def relatorio_atrasos():
    """Regra: 'o técnico precisa de um relatório dos atrasos'. Cobre empréstimos em aberto e atrasados."""
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT E.ID, A.NOME, Q.NOME, E.DATA_PREVISTA,
               julianday('now') - julianday(E.DATA_PREVISTA) AS DIAS_ATRASO
        FROM EMPRESTIMO E
        JOIN ALUNO A ON A.ID = E.ALUNO_ID
        JOIN EQUIPAMENTO Q ON Q.ID = E.EQUIPAMENTO_ID
        WHERE E.DATA_DEVOLUCAO IS NULL
          AND E.DATA_PREVISTA < date('now')
        ORDER BY DIAS_ATRASO DESC;
    """)
    linhas = cursor.fetchall()
    conn.close()

    print("\n--- RELATÓRIO DE ATRASOS (uso do técnico) ---")
    if not linhas:
        print("Nenhum atraso em aberto no momento.")
    for id_, aluno, equip, data_prev, dias_atraso in linhas:
        print(f"[Empréstimo {id_}] {aluno} está com '{equip}' desde o prazo {data_prev} | {int(dias_atraso)} dia(s) de atraso")


def listar_alunos_com_pendencia():
    """Varre todos os alunos e mostra quais têm pendência e por quê (as 2 regras juntas)."""
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("SELECT ID, NOME FROM ALUNO ORDER BY ID;")
    alunos = cursor.fetchall()
    conn.close()

    print("\n--- ALUNOS COM PENDÊNCIA ---")
    encontrou = False
    for id_aluno, nome in alunos:
        tem_pendencia, motivo = aluno_tem_pendencia(id_aluno)
        if tem_pendencia:
            encontrou = True
            print(f"[{id_aluno}] {nome} -> {motivo}")
    if not encontrou:
        print("Nenhum aluno com pendência no momento.")

<h3>Deleção de algum cadastro dos armazenados atualmente</h3>

In [31]:
def deletar_emprestimo(id_emprestimo):
    conn = conectar()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM EMPRESTIMO WHERE ID = ?;", (id_emprestimo,))
    conn.commit()
    removido = cursor.rowcount
    conn.close()
    if removido:
        print(f"Empréstimo {id_emprestimo} removido do sistema.")
    else:
        print(f"Nenhum empréstimo encontrado com ID {id_emprestimo}.")

<h3>Inserindo dados de teste</h3>

In [32]:
# Inserindo alunos de teste
inserir_aluno("Ana Souza")
inserir_aluno("Bruno Lima")
inserir_aluno("Carla Mendes")

# Inserindo equipamentos de teste
inserir_equipamento("Notebook Dell")
inserir_equipamento("Projetor Epson")
inserir_equipamento("Câmera Canon")
inserir_equipamento("Microfone Shure")
inserir_equipamento("Tablet Samsung")

Aluno 'Ana Souza' cadastrado com ID 1.
Aluno 'Bruno Lima' cadastrado com ID 2.
Aluno 'Carla Mendes' cadastrado com ID 3.
Equipamento 'Notebook Dell' cadastrado com ID 1.
Equipamento 'Projetor Epson' cadastrado com ID 2.
Equipamento 'Câmera Canon' cadastrado com ID 3.
Equipamento 'Microfone Shure' cadastrado com ID 4.
Equipamento 'Tablet Samsung' cadastrado com ID 5.


5

<h3>Testes para o sistema de empréstimos</h3>

In [33]:
print("== Teste 1: empréstimo válido ==")
emprestar_equipamento(1, 1)   # Ana pega Notebook Dell

print("\n== Teste 2: aluno inexistente ==")
emprestar_equipamento(99, 2)  # deve falhar

print("\n== Teste 3: equipamento inexistente ==")
emprestar_equipamento(1, 99)  # deve falhar

print("\n== Teste 4: equipamento já emprestado ==")
emprestar_equipamento(2, 1)   # Notebook Dell já está com a Ana -> deve falhar

listar_emprestimos_ativos()

== Teste 1: empréstimo válido ==
Empréstimo 1 registrado: aluno 1 pegou equipamento 1. Prazo: 7 dias.

== Teste 2: aluno inexistente ==
Erro: aluno com ID 99 não existe.

== Teste 3: equipamento inexistente ==
Erro: equipamento com ID 99 não existe.

== Teste 4: equipamento já emprestado ==
Erro: o equipamento 1 já está emprestado a outro aluno.

--- EQUIPAMENTOS EMPRESTADOS NO MOMENTO ---
[Empréstimo 1] Notebook Dell está com Ana Souza | pegou 2026-08-11 | previsto 2026-08-18


In [34]:
conn = conectar()
cursor = conn.cursor()
cursor.execute("""
    INSERT INTO EMPRESTIMO (ALUNO_ID, EQUIPAMENTO_ID, DATA_EMPRESTIMO, DATA_PREVISTA, DATA_DEVOLUCAO)
    VALUES (2, 2, date('now', '-10 days'), date('now', '-3 days'), NULL);
""")
conn.commit()
conn.close()
print("Empréstimo de teste criado: Bruno pegou o Projetor Epson há 10 dias, prazo venceu há 3 dias.")

Empréstimo de teste criado: Bruno pegou o Projetor Epson há 10 dias, prazo venceu há 3 dias.


In [35]:
print("== Teste 5: relatório de atrasos deve mostrar o Bruno ==")
relatorio_atrasos()

print("\n== Teste 6: alunos com pendência deve mostrar o Bruno (pendência tipo 1) ==")
listar_alunos_com_pendencia()

print("\n== Teste 7: Bruno tenta pegar outro equipamento -> deve ser BLOQUEADO ==")
emprestar_equipamento(2, 3)

== Teste 5: relatório de atrasos deve mostrar o Bruno ==

--- RELATÓRIO DE ATRASOS (uso do técnico) ---
[Empréstimo 2] Bruno Lima está com 'Projetor Epson' desde o prazo 2026-08-08 | 3 dia(s) de atraso

== Teste 6: alunos com pendência deve mostrar o Bruno (pendência tipo 1) ==

--- ALUNOS COM PENDÊNCIA ---
[2] Bruno Lima -> possui empréstimo(s) em aberto e atrasado(s): empréstimo 2 (equip. 2, previsto 2026-08-08)

== Teste 7: Bruno tenta pegar outro equipamento -> deve ser BLOQUEADO ==
Erro: aluno 2 não pode pegar equipamento — possui empréstimo(s) em aberto e atrasado(s): empréstimo 2 (equip. 2, previsto 2026-08-08).


False

In [36]:
conn = conectar()
cursor = conn.cursor()
cursor.execute("""
    INSERT INTO EMPRESTIMO (ALUNO_ID, EQUIPAMENTO_ID, DATA_EMPRESTIMO, DATA_PREVISTA, DATA_DEVOLUCAO)
    VALUES (3, 4, date('now', '-9 days'), date('now', '-2 days'), NULL);
""")
conn.commit()
id_emprestimo_carla = cursor.lastrowid
conn.close()
print(f"Empréstimo de teste criado: Carla pegou o Microfone Shure há 9 dias, prazo venceu há 2 dias (ID {id_emprestimo_carla}).")

Empréstimo de teste criado: Carla pegou o Microfone Shure há 9 dias, prazo venceu há 2 dias (ID 3).


In [37]:
print("== Teste 8: devolução em atraso deve bloquear a Carla por 7 dias ==")
devolver_equipamento(id_emprestimo_carla)

print("\n== Teste 9: alunos com pendência agora deve mostrar Bruno E Carla ==")
listar_alunos_com_pendencia()

print("\n== Teste 10: Carla tenta pegar outro equipamento -> deve ser BLOQUEADA mesmo sem empréstimo em aberto ==")
emprestar_equipamento(3, 5)

== Teste 8: devolução em atraso deve bloquear a Carla por 7 dias ==
Devolução do empréstimo 3 registrada COM ATRASO (previsto: 2026-08-09).
Aluno 3 está bloqueado para novos empréstimos pelos próximos 7 dias.

== Teste 9: alunos com pendência agora deve mostrar Bruno E Carla ==

--- ALUNOS COM PENDÊNCIA ---
[2] Bruno Lima -> possui empréstimo(s) em aberto e atrasado(s): empréstimo 2 (equip. 2, previsto 2026-08-08)
[3] Carla Mendes -> está bloqueado até 2026-08-18 por ter devolvido um item com atraso

== Teste 10: Carla tenta pegar outro equipamento -> deve ser BLOQUEADA mesmo sem empréstimo em aberto ==
Erro: aluno 3 não pode pegar equipamento — está bloqueado até 2026-08-18 por ter devolvido um item com atraso.


False

In [38]:
print("== Teste 11: devolução dentro do prazo não deve gerar bloqueio ==")
devolver_equipamento(1)  # Ana devolve Notebook Dell dentro do prazo

print("\n== Teste 12: Ana pode pegar outro item normalmente ==")
emprestar_equipamento(1, 5)

== Teste 11: devolução dentro do prazo não deve gerar bloqueio ==
Devolução do empréstimo 1 registrada dentro do prazo.

== Teste 12: Ana pode pegar outro item normalmente ==
Empréstimo 4 registrado: aluno 1 pegou equipamento 5. Prazo: 7 dias.


4

<h3>Menu inicial</h3>

In [39]:
from IPython.display import clear_output

_menu_rodando = False

def pausar():
    input("\nPressione ENTER para voltar ao menu...")

def menu():
    global _menu_rodando
    if _menu_rodando:
        print("O menu já está em execução em outra chamada. Não inicie de novo.")
        return
    _menu_rodando = True

    try:
        while True:
            clear_output(wait=True)
            print("===== SISTEMA DE EMPRÉSTIMOS - LABORATÓRIO =====")
            print("--- ALUNOS ---")
            print("1  - Listar alunos")
            print("2  - Exibir aluno por ID")
            print("3  - Cadastrar aluno")
            print("4  - Atualizar aluno")
            print("5  - Remover aluno")
            print("--- EQUIPAMENTOS ---")
            print("6  - Listar equipamentos")
            print("7  - Exibir equipamento por ID")
            print("8  - Cadastrar equipamento")
            print("9  - Atualizar equipamento")
            print("10 - Remover equipamento")
            print("--- EMPRÉSTIMOS ---")
            print("11 - Listar histórico de empréstimos")
            print("12 - Listar empréstimos ativos (o que está emprestado e para quem)")
            print("13 - Exibir empréstimo por ID")
            print("14 - Registrar empréstimo (aluno pega equipamento)")
            print("15 - Registrar devolução")
            print("16 - Remover empréstimo (correção administrativa)")
            print("17 - Relatório de atrasos (técnico)")
            print("18 - Listar alunos com pendência")
            print("--- ---")
            print("0  - Sair")

            try:
                opcao = input("Escolha uma opção: ")

                if opcao == "1":
                    listar_alunos(); pausar()
                elif opcao == "2":
                    buscar_aluno_por_id(int(input("ID do aluno: "))); pausar()
                elif opcao == "3":
                    inserir_aluno(input("Nome do aluno: ")); pausar()
                elif opcao == "4":
                    id_a = int(input("ID do aluno a atualizar: "))
                    atualizar_aluno(id_a, input("Novo nome: ")); pausar()
                elif opcao == "5":
                    deletar_aluno(int(input("ID do aluno a remover: "))); pausar()

                elif opcao == "6":
                    listar_equipamentos(); pausar()
                elif opcao == "7":
                    buscar_equipamento_por_id(int(input("ID do equipamento: "))); pausar()
                elif opcao == "8":
                    inserir_equipamento(input("Nome do equipamento: ")); pausar()
                elif opcao == "9":
                    id_e = int(input("ID do equipamento a atualizar: "))
                    atualizar_equipamento(id_e, input("Novo nome: ")); pausar()
                elif opcao == "10":
                    deletar_equipamento(int(input("ID do equipamento a remover: "))); pausar()

                elif opcao == "11":
                    listar_emprestimos(); pausar()
                elif opcao == "12":
                    listar_emprestimos_ativos(); pausar()
                elif opcao == "13":
                    buscar_emprestimo_por_id(int(input("ID do empréstimo: "))); pausar()
                elif opcao == "14":
                    id_aluno = int(input("ID do aluno: "))
                    id_equip = int(input("ID do equipamento: "))
                    emprestar_equipamento(id_aluno, id_equip); pausar()
                elif opcao == "15":
                    devolver_equipamento(int(input("ID do empréstimo a devolver: "))); pausar()
                elif opcao == "16":
                    deletar_emprestimo(int(input("ID do empréstimo a remover: "))); pausar()
                elif opcao == "17":
                    relatorio_atrasos(); pausar()
                elif opcao == "18":
                    listar_alunos_com_pendencia(); pausar()

                elif opcao == "0":
                    print("Encerrando o sistema.")
                    break
                else:
                    print("Opção inválida, tente novamente.")
                    pausar()

            except ValueError:
                print("Entrada inválida. Digite apenas números onde for pedido um ID.")
                pausar()
            except Exception as e:
                import traceback
                print(f"Erro: {type(e).__name__} - {e}")
                traceback.print_exc()
                pausar()
    finally:
        _menu_rodando = False


menu()

===== SISTEMA DE EMPRÉSTIMOS - LABORATÓRIO =====
--- ALUNOS ---
1  - Listar alunos
2  - Exibir aluno por ID
3  - Cadastrar aluno
4  - Atualizar aluno
5  - Remover aluno
--- EQUIPAMENTOS ---
6  - Listar equipamentos
7  - Exibir equipamento por ID
8  - Cadastrar equipamento
9  - Atualizar equipamento
10 - Remover equipamento
--- EMPRÉSTIMOS ---
11 - Listar histórico de empréstimos
12 - Listar empréstimos ativos (o que está emprestado e para quem)
13 - Exibir empréstimo por ID
14 - Registrar empréstimo (aluno pega equipamento)
15 - Registrar devolução
16 - Remover empréstimo (correção administrativa)
17 - Relatório de atrasos (técnico)
18 - Listar alunos com pendência
--- ---
0  - Sair
Encerrando o sistema.
